# 웹 문서 기반 RAG 구축하기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- `WebBaseLoader`로 웹 페이지를 크롤링해 Document 객체로 변환하기
- `BeautifulSoup`의 `SoupStrainer`로 HTML에서 필요한 영역만 추출하기
- 웹 문서의 특성(HTML 노이즈, 긴 텍스트)을 고려한 전처리 적용하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인 이해
- HTML 기본 구조 (태그, 클래스, id 등)

---

```mermaid
flowchart LR
    URL[웹 페이지 URL]:::input --> WL[WebBaseLoader<br/>HTTP 요청]:::process
    WL --> BS[BeautifulSoup<br/>HTML 파싱]:::process
    BS --> SS[SoupStrainer<br/>본문 영역만 추출]:::process
    SS --> D[Document 객체]:::process
    D --> RAG[RAG 파이프라인<br/>분할 → 임베딩 → 검색]:::process
    RAG --> A[웹 기반<br/>질의응답]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## PDF vs 웹 문서

| 특성 | PDF | 웹 문서 |
|------|-----|---------|
| 업데이트 | 파일 교체 필요 | URL 그대로, 자동 반영 |
| 구조 | 고정된 레이아웃 | HTML 태그 파싱 필요 |
| 접근성 | 파일 배포 필요 | URL만 있으면 즉시 |
| 노이즈 | 적음 | 네비게이션·광고 제거 필요 |

> **실무 팁**: `SoupStrainer`에 Wikipedia의 `mw-parser-output` 같은 본문 클래스를 지정하면 불필요한 HTML 요소를 효과적으로 제거할 수 있어요.

> 🎯 **강의 포인트**: 이 노트북의 핵심은 **"로더만 바꾸면 RAG 파이프라인은 동일하다"**는 점입니다. `PyMuPDFLoader` → `WebBaseLoader`로 교체하는 것 외에는 00-RAG-Basic과 완전히 동일합니다. RAG 파이프라인의 모듈성과 교체 가능성을 보여주는 최적의 예시입니다.

> ⚠️ **자주 하는 실수**: `SoupStrainer` 없이 전체 HTML을 로드하면 네비게이션 바, 사이드바, 광고 등 노이즈가 문서에 포함됩니다. 반드시 `parse_only` 파라미터로 본문 영역만 지정하세요.

## 학습 목표

- `WebBaseLoader`를 사용한 웹 페이지 크롤링
- BeautifulSoup을 활용한 HTML 파싱
- 웹 문서의 특성을 고려한 전처리
- 웹 기반 RAG 시스템 구축 및 테스트

## 웹 문서 vs PDF 문서

### 웹 문서의 특징

**장점**:
- ✅ 실시간 업데이트: 최신 정보 반영 가능
- ✅ 접근성: URL만 있으면 즉시 접근
- ✅ 구조화: HTML 태그로 구조 파악 용이

**주의사항**:
- ⚠️ HTML 노이즈: 네비게이션, 광고 등 불필요한 요소 제거 필요
- ⚠️ 동적 콘텐츠: JavaScript로 생성되는 내용은 추가 처리 필요
- ⚠️ 웹 크롤링 정책: robots.txt 및 이용 약관 확인 필수

### PDF 문서의 특징

**장점**:
- ✅ 안정성: 내용 변경 없음
- ✅ 완결성: 전체 문서가 하나의 파일
- ✅ 형식 보존: 레이아웃과 서식 유지

**주의사항**:
- ⚠️ 업데이트: 수동으로 파일 교체 필요
- ⚠️ 파싱 난이도: 복잡한 레이아웃 처리 어려움

## 환경 설정

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

## 실습 웹사이트

이번 실습에서는 **Wikipedia 한국어 페이지**를 사용합니다.

- 주제: 인공지능 (Artificial Intelligence)
- URL: https://ko.wikipedia.org/wiki/인공지능
- 선택 이유: 안정적이고 구조화된 콘텐츠, 풍부한 정보

**학습 목표**: 웹 페이지를 크롤링하여 인공지능에 대한 질문에 답변하는 RAG 시스템 구축

## 필수 라이브러리 import

**웹 크롤링을 위한 추가 라이브러리**:
- `WebBaseLoader`: 웹 페이지 로딩
- `bs4` (BeautifulSoup): HTML 파싱 및 필터링

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# ---------------------------------------------------
# WebBaseLoader로 웹 페이지 크롤링 — 본문 영역만 추출
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 본문을 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={...})}
# 예상 결과: 로드된 문서 수와 문자 길이 출력
# ============================================================

# 단계 1: 웹 페이지 로드
# 🎯 강의 포인트: SoupStrainer로 원하는 HTML 영역만 파싱 — 속도·메모리·품질 모두 개선
loader = WebBaseLoader(
    web_paths=("https://ko.wikipedia.org/wiki/인공지능",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"id": "bodyContent"}  # ⚠️ 주의: 사이트마다 본문 클래스가 다름
        )
    ),
)

docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print(f"문서 길이: {len(docs[0].page_content)} 문자")
print(f"\n문서 미리보기 (첫 300자):\n{docs[0].page_content[:300]}...")

로드된 문서 수: 1
문서 길이: 51287 문자

문서 미리보기 (첫 300자):




위키백과, 우리 모두의 백과사전.


 AI는 여기로 연결됩니다. 조류가 걸리는 호흡기 질병에 대해서는 조류 인플루엔자 문서를, 다른 뜻에 대해서는 AI (동음이의) 문서를 참고하십시오.
 위키백과의 인공지능에 대해서는 위키백과:인공지능 문서를 참고하십시오.

시리즈의 일부인공지능 (AI)
주요 목표
인공 일반 지능
지능형 에이전트
재귀적 자기 개선
계획 및 스케줄링
컴퓨터 비전
일반 게임 플레이
지식 표현 및 추론
자연어 처리
로봇공학
AI 안전

접근 방식
기계 학습
기호주의
딥 러닝
베이즈 네트워크
진화 알고리즘
하이브...


웹 페이지의 메타데이터를 확인합니다. URL, 제목 등의 정보가 포함되어 있습니다.

In [5]:
# 메타데이터 확인
print("=" * 60)
print("[문서 메타데이터]")
print("=" * 60)
for key, value in docs[0].metadata.items():
    print(f"{key}: {value}")

[문서 메타데이터]
source: https://ko.wikipedia.org/wiki/인공지능


### 2단계: 문서 분할 (Text Splitting)

웹 페이지는 일반적으로 하나의 긴 문서로 로드되므로 적절한 크기로 분할이 필요합니다.

**웹 문서 분할 시 고려사항**:
- 단락 구분이 명확하지 않을 수 있음
- HTML 태그 제거 후 공백 처리 필요
- 제목과 본문의 연결성 유지

In [6]:
# 단계 2: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200  # 웹 문서는 문맥 연결이 중요하므로 overlap을 크게
)
split_documents = text_splitter.split_documents(docs)

print(f"원본 문서 수: {len(docs)}")
print(f"분할된 청크 수: {len(split_documents)}")
if not split_documents:
    print("⚠️ 웹 문서 길이가 짧아 청크가 생성되지 않았습니다. chunk_size를 조정해 보세요.")
else:
    print(f"\n첫 번째 청크 내용:\n{split_documents[0].page_content}")


원본 문서 수: 1
분할된 청크 수: 68

첫 번째 청크 내용:
위키백과, 우리 모두의 백과사전.


 AI는 여기로 연결됩니다. 조류가 걸리는 호흡기 질병에 대해서는 조류 인플루엔자 문서를, 다른 뜻에 대해서는 AI (동음이의) 문서를 참고하십시오.
 위키백과의 인공지능에 대해서는 위키백과:인공지능 문서를 참고하십시오.

시리즈의 일부인공지능 (AI)
주요 목표
인공 일반 지능
지능형 에이전트
재귀적 자기 개선
계획 및 스케줄링
컴퓨터 비전
일반 게임 플레이
지식 표현 및 추론
자연어 처리
로봇공학
AI 안전

접근 방식
기계 학습
기호주의
딥 러닝
베이즈 네트워크
진화 알고리즘
하이브리드 인텔리전트 시스템
시스템 통합
오픈 소스 인공지능
AI 데이터 센터

애플리케이션
생물정보학
딥페이크
지구과학
금융
생성형 인공지능
예술
오디오
음악
정부
의료
정신 건강
산업
소프트웨어 개발
번역
군사
물리학
프로젝트

철학
AI 정렬
인공 의식
쓰라린 교훈
중국어 방
친절한 AI
윤리
실존적 위험
튜링 검사
불쾌한 골짜기
인간-AI 상호작용

역사
연표
진행상황
AI 겨울
AI 붐
인공지능 거품

논란
딥페이크 포르노그래피
테일러 스위프트 딥페이크 포르노그래피 논란
Grok 딥페이크 포르노그래피 논란
구글 제미나이 이미지 생성 논란
거대 AI 실험 중단: 공개 서한
오픈AI의 샘 올트먼 해임 사건
AI 위험에 관한 성명
테이 (챗봇)
Théâtre D'opéra Spatial
보이스버스 NFT 표절 스캔들


### 3-4단계: 임베딩 및 벡터 저장

PDF와 동일한 방식으로 임베딩 모델을 생성하고 벡터스토어에 저장합니다.

In [7]:
# ---------------------------------------------------
# OpenAI 임베딩으로 웹 문서를 FAISS 벡터스토어에 저장
# ---------------------------------------------------
# ============================================================
# TODO: split_documents를 임베딩하여 FAISS 벡터스토어를 만드세요
# 힌트: OpenAIEmbeddings(model="text-embedding-3-small"), FAISS.from_documents(...)
# 예상 결과: "벡터스토어 생성 완료" 출력
# ============================================================

# 단계 3: 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if not split_documents:
    vectorstore = None
    print("⚠️ 청크가 없어 벡터스토어를 생성하지 않습니다.")
else:
    # 단계 4: 벡터스토어 생성
    vectorstore = FAISS.from_documents(
        documents=split_documents,
        embedding=embeddings
    )

    print(f"벡터스토어 생성 완료")
    print(f"저장된 청크 수: {len(split_documents)}")

벡터스토어 생성 완료
저장된 청크 수: 68


웹 문서에서 특정 키워드로 검색이 잘 되는지 테스트합니다.

In [8]:
# 벡터스토어 검색 테스트
test_query = "기계 학습"
if vectorstore is None:
    print("⚠️ 벡터스토어가 없어 검색 테스트를 건너뜁니다.")
else:
    search_results = vectorstore.similarity_search(test_query, k=3)

    print(f"검색어: '{test_query}'")
    print("=" * 60)
    for i, doc in enumerate(search_results, 1):
        print(f"\n[검색 결과 {i}]")
        print(doc.page_content[:250] + "...")
        print(f"출처: {doc.metadata.get('source', 'N/A')}")


검색어: '기계 학습'

[검색 결과 1]
특히 1980년대에 들어서 Back propagation (인공지능 학습방법: Training Method)가 소개되면서 많은 연구가 진행되었음에도, 신경망을 이용한 인공지능은 아직 초보단계이다. 인공신경망 (Artificial Neural Networks)을 이용한 많은 연구가 현재에도 진행되고 있지만, 몇 가지 장애로 인해서 실용화하기엔 아직도 먼 기술이다. 인공신경망을 이용한 인공지능이 어느 정도 실용화되기 위해선 우선 실효성 있는 학습방...
출처: https://ko.wikipedia.org/wiki/인공지능

[검색 결과 2]
튜링 테스트
1950년 앨런 튜링은 생각하는 기계의 구현 가능성에 대한 분석이 담긴, 인공지능 역사에서 혁혁한 논문을 발표했다.[17] 그는 "생각"을 정의하기 어려움에 주목해, 그 유명한 튜링테스트를 고안했다. 텔레프린터를 통한 대화에서 기계가 사람인지 기계인지 구별할 수 없을 정도로 대화를 잘 이끌어 간다면, 이것은 기계가 "생각"하고 있다고 말할 충분한 근거가 된다는 것이었다.[18] 튜링 테스트는 인공 지능에 대한 최초의 심도 있는 철학...
출처: https://ko.wikipedia.org/wiki/인공지능

[검색 결과 3]
인공지능의 이론적인 결과
어떤 사람들은 현재 알려진 어떤 시스템보다도 지능적이며 복잡한 시스템의 등장을 예견하기도 한다. 이와 같은 가상적인 시스템들을 '비결정적인 인공지능 시스템'의 약자인 atilect라고 한다. 이와 같은 시스템이 만들어진다면 그동안 인류에게 문제시되지 않았던 많은 윤리적인 문제들이 발생하게 된다.
이에 대한 토론은 시간이 흐름에 따라 '가능성'보다는 '의도'에 점점 초점을 맞추게 되었다. 이러한 초점의 이동은 휴고 더개리...
출처: https://ko.wikipedia.org/wiki/인공지능


### 5-8단계: 검색기, 프롬프트, LLM, 체인 생성

이후 단계는 PDF 기반 RAG와 동일합니다. 검색기를 생성하고 RAG 체인을 구축합니다.

In [9]:
# 단계 5: 검색기 생성
if vectorstore is None:
    retriever = None
    print("⚠️ 벡터스토어가 없어 검색기를 생성하지 않습니다.")
else:
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    print("검색기 생성 완료")


검색기 생성 완료


In [10]:
# 단계 6: 프롬프트 템플릿
prompt = PromptTemplate.from_template(
    """당신은 웹 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 웹 페이지 내용(Context)을 바탕으로 질문(Question)에 답변해 주세요.

답변 시 주의사항:
- 웹 문서에서 찾을 수 있는 정보만 사용하세요
- 문서에 없는 내용이라면 "제공된 웹 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요
- 답변은 명확하고 이해하기 쉽게 작성하세요
- 한국어로 답변하세요

#Context:
{context}

#Question:
{question}

#Answer:"""
)

print("프롬프트 템플릿 생성 완료")

프롬프트 템플릿 생성 완료


In [12]:
# ---------------------------------------------------
# LCEL로 RAG 체인 조립 — 검색기 + 프롬프트 + LLM 연결
# ---------------------------------------------------
# ============================================================
# TODO: retriever, prompt, llm을 LCEL 파이프라인으로 연결하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "RAG 체인 생성 완료" 출력
# ============================================================

if retriever is None:
    rag_chain = None
    print("⚠️ 검색기가 없어 RAG 체인을 생성하지 않습니다.")
else:
    # 단계 8: RAG 체인 생성
    # 💡 팁: PDF RAG와 체인 구조가 완전히 동일 — 로더만 교체하면 모든 소스에 재사용 가능
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG 체인 생성 완료")

RAG 체인 생성 완료


## 웹 기반 RAG 실행 및 테스트

이제 완성된 RAG 체인을 사용하여 인공지능 Wikipedia 페이지에 대한 질문에 답변해봅니다.

In [14]:
# 질문 1: 기본 개념 질의
question1 = "인공지능이란 무엇인가요?"

if rag_chain is None:
    print("⚠️ RAG 체인이 없어 답변을 생성할 수 없습니다.")
else:

    print(f"질문: {question1}")
    print("=" * 60)
    answer1 = rag_chain.invoke(question1)
    print(f"답변:\n{answer1}")

질문: 인공지능이란 무엇인가요?
답변:
인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야입니다. 이는 인간을 포함한 동물이 갖고 있는 자연 지능과는 다른 개념으로, 인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템을 의미합니다. 일반적으로 범용 컴퓨터에 적용되며, 이러한 지능을 만들 수 있는 방법론이나 실현 가능성을 연구하는 과학 기술 분야를 포함합니다.


In [15]:
# 질문 2: 구체적인 정보 질의
question2 = "기계 학습과 딥러닝의 차이점은 무엇인가요?"

if rag_chain is None:
    print("⚠️ RAG 체인이 없어 답변을 생성할 수 없습니다.")
else:

    print(f"질문: {question2}")
    print("=" * 60)
    answer2 = rag_chain.invoke(question2)
    print(f"답변:\n{answer2}")

질문: 기계 학습과 딥러닝의 차이점은 무엇인가요?
답변:
제공된 웹 문서에서 해당 정보를 찾을 수 없습니다.


In [16]:
# 질문 3: 역사적 정보 질의
question3 = "인공지능의 역사에서 중요한 사건은 무엇인가요?"

if rag_chain is None:
    print("⚠️ RAG 체인이 없어 답변을 생성할 수 없습니다.")
else:

    print(f"질문: {question3}")
    print("=" * 60)
    answer3 = rag_chain.invoke(question3)
    print(f"답변:\n{answer3}")

질문: 인공지능의 역사에서 중요한 사건은 무엇인가요?
답변:
인공지능의 역사에서 중요한 사건으로는 다음과 같은 것들이 있습니다:

1. **전문가 시스템의 발전**: 1980년대에 전문가 시스템이 인공지능 프로그램의 형태로 전 세계적으로 사용되었으며, 일본 정부는 5세대 컴퓨터 프로젝트에 적극적으로 투자하였습니다. 이 시기에 존 홉필드와 데이비드 루멜하트의 신경망 이론이 복원되었습니다.

2. **MYCIN의 개발**: 1972년에 개발된 MYCIN은 전염되는 혈액 질환을 진단하는 전문가 시스템으로, 인공지능의 초기 성공 사례 중 하나입니다.

3. **XCON 시스템**: 1980년에 디지털 장비 회사인 CMU에서 완성된 XCON 시스템은 매년 4천만 달러를 절약시켜주는 성과를 나타냈습니다.

4. **체스에서의 딥 블루**: 1997년, 체스를 두는 컴퓨터인 딥 블루가 세계 체스 챔피언 가리 카스파로프를 물리친 사건은 인공지능의 성능을 대중에게 알리는 중요한 계기가 되었습니다.

이 외에도 인공지능 연구의 기금 지원과 그에 따른 연구의 침체기인 '인공지능의 겨울' 등도 중요한 역사적 사건으로 언급될 수 있습니다.


## 전체 코드 (통합 버전)

In [17]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

try:
    # 1. 웹 페이지 로드
    loader = WebBaseLoader(
        web_paths=("https://ko.wikipedia.org/wiki/인공지능",),
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(
                "div",
                attrs={"id": "bodyContent"}
            )
        ),
    )
    docs = loader.load()
except Exception as exc:
    docs = []
    print(f"⚠️ 웹 페이지를 불러오지 못했습니다: {exc}")

if not docs:
    print("⚠️ 샘플 RAG 체인을 실행할 수 없어 단계를 건너뜁니다.")
else:
    # 2. 문서 분할
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    split_documents = text_splitter.split_documents(docs)

    if not split_documents:
        print("⚠️ 웹 문서 청크가 비어 RAG 체인을 생성하지 않습니다.")
    else:
        # 3. 임베딩 생성
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # 4. 벡터스토어 생성
        vectorstore = FAISS.from_documents(
            documents=split_documents,
            embedding=embeddings
        )

        # 5. 검색기 생성
        retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

        # 6. 프롬프트
        prompt = PromptTemplate.from_template(
            """당신은 웹 문서 기반 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 웹 페이지 내용(Context)을 바탕으로 질문(Question)에 답변해 주세요.

답변 시 주의사항:
- 웹 문서에서 찾을 수 있는 정보만 사용하세요
- 문서에 없는 내용이라면 "제공된 웹 문서에서 해당 정보를 찾을 수 없습니다"라고 답하세요

#Context:
{context}

#Question:
{question}

#Answer:
"""
        )

        # 7. LLM
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        # 8. RAG 체인
        rag_chain = (
            {"context": retriever, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )

        # 실행
        question = "인공지능이란 무엇인가요?"
        answer = rag_chain.invoke(question)
        print(f"질문: {question}")
        print(f"답변: {answer}")


질문: 인공지능이란 무엇인가요?
답변: 인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나입니다. 이는 인간을 포함한 동물이 갖고 있는 지능인 자연 지능(natural intelligence)과는 다른 개념으로, 인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템을 의미합니다. 일반적으로 범용 컴퓨터에 적용된다고 가정하며, 이러한 지능을 만들 수 있는 방법론이나 실현 가능성을 연구하는 과학 기술 분야를 지칭하기도 합니다.


> 💡 **실무 팁**: 웹 크롤링 기반 RAG를 프로덕션에 적용할 때는 크롤링 결과를 캐시(파일 또는 DB)에 저장하세요. 매 요청마다 웹을 크롤링하면 느리고 외부 사이트 장애에 취약합니다. 주기적 크롤링 + 벡터스토어 갱신 패턴을 권장합니다.

## 💡 핵심 정리

### 웹 기반 RAG의 특징

1. **WebBaseLoader**: URL만으로 간단히 문서 로드
2. **BeautifulSoup**: HTML 파싱으로 필요한 부분만 추출
3. **실시간성**: 웹 페이지 업데이트 시 즉시 반영 가능
4. **접근성**: 파일 다운로드 없이 URL로 접근

### PDF vs 웹 선택 가이드

| 상황 | 추천 소스 | 이유 |
|------|----------|------|
| **최신 정보 필요** | 웹 | 실시간 업데이트 반영 |
| **안정적인 참조** | PDF | 내용 변경 없음 |
| **빠른 프로토타입** | 웹 | URL만으로 즉시 시작 |
| **법률/의료 문서** | PDF | 정확한 형식 보존 필요 |
| **뉴스/블로그** | 웹 | 자주 업데이트되는 콘텐츠 |

### 주의사항

- ⚠️ **웹 크롤링 정책**: robots.txt 확인 및 이용 약관 준수
- ⚠️ **HTML 노이즈**: SoupStrainer로 본문만 추출
- ⚠️ **동적 콘텐츠**: JavaScript 렌더링이 필요한 경우 Selenium 사용 고려
- ⚠️ **인코딩**: 한글 깨짐 방지를 위한 인코딩 설정 확인

### 다음 단계

다음 노트북에서는 RAG 성능을 향상시키는 고급 기법들을 다룹니다:
- Ensemble Retriever (벡터 + 키워드 검색 결합)
- Reranking (검색 결과 재정렬)
- Query Transformation (질문 재작성)

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 도구 | `WebBaseLoader` + `BeautifulSoup` |
| HTML 필터링 | `SoupStrainer`로 원하는 태그(`div`, `p` 등)만 추출 |
| 로더 파라미터 | `bs_kwargs={"parse_only": SoupStrainer(...)}` |
| 실시간 정보 | URL 기반이므로 웹 콘텐츠가 갱신될 때마다 최신 정보 반영 |
| 주의 | 로그인이 필요하거나 JS 렌더링 페이지는 바로 사용 불가 |

### 웹 기반 RAG vs PDF 기반 RAG 비교

| 항목 | 웹 기반 | PDF 기반 |
|------|---------|----------|
| 로더 | `WebBaseLoader` | `PyPDFLoader` |
| 정보 최신성 | 실시간 | 파일 업로드 시점 |
| 접근 방식 | URL | 로컬 파일 경로 |
| HTML 파싱 | BeautifulSoup 필요 | 자동 처리 |
| 적합한 상황 | 뉴스, 위키, 공식 문서 | 보고서, 논문, 매뉴얼 |

### 다음 노트북 예고

**02-RAG-Advanced** — EnsembleRetriever(BM25+FAISS)와 MMR(최대 한계 관련성)을 결합해 기본 RAG보다 더 다양하고 관련성 높은 문서를 검색하는 방법을 배워요.